# APK 4720 / APK 6725 — Assignment 5

Submit as a Jupyter notebook (.ipynb) named `YourName_Assignment_5`.

This assignment assesses your ability to use **Pandas** and **Matplotlib** to
organise and visualise data. You will work with running biomechanics from a
treadmill study: 28 volunteers ran at three speeds — **2.5, 3.5 and 4.5 m/s** —
while motion capture and force plates recorded their joints.

> Fukuchi, R. K., Fukuchi, C. A., & Duarte, M. (2017). A public dataset of
> running biomechanics and the effects of running speed on lower extremity
> kinematics and kinetics. *PeerJ* 5, e3298.

Each participant has one file with columns named `[Side][Joint][Var][Axis][Speed]`:

* Side: `R` or `L`
* Joint: `hip`, `knee`, `ankle`
* Var: `Ang` (angle, deg), `Mom` (moment, Nm/kg), `grf` (ground reaction force,
  N/kg), `Pow` (power, W/kg — no axis)
* Axis: `X`, `Y`, `Z`
* Speed: `25`, `35`, `45` (i.e. 2.5, 3.5, 4.5 m/s)

So `RkneeAngZ35` is the right knee angle about the Z axis at 3.5 m/s, and
`RanklePow45` is the right ankle power at 4.5 m/s. The `PercGcycle` column is the
percent of the gait cycle (0–100), so every curve is 101 rows.

**Which axis is which?** For these joints the **Z axis is the sagittal plane** —
flexion and extension, the large motion you see when someone runs. That is the
axis this assignment focuses on. (X and Y are the smaller side-to-side and
rotation components.)

Unzip `gaitData.zip` next to this notebook. You get a `gaitData/` folder with
`RBDS001processed.txt` … `RBDS028processed.txt` and `RBDSinfo.csv`.

Total = 5 points

## Run this first

Run the cell below, at the start of every session. It downloads the course files to your Google Drive and keeps them up to date. When you see "Setup complete", you are ready to work.

**Do not edit or save your work inside the `AIforMovementScience` folder.** Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
import os, sys, zipfile

# [course-setup] Run this cell first, every session (or use Runtime > Run all)
os.chdir('/content')                      # start from a valid directory
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/drive/MyDrive/AIforMovementScience'
if os.path.isdir(REPO + '/.git'):
    # already downloaded before: just update it
    !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
else:
    # first time, or a broken/partial copy: download a fresh copy
    !rm -rf "$REPO"
    !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"

WORKDIR = REPO + '/Assignments'          # this notebook's folder in the course files
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)           # so local .py files (e.g. FunctionExamples.py) import correctly
DATA = REPO + '/lectures/data'        # the folder that holds this assignment's data files
print('Setup complete. You can now run the rest of the notebook.')


#now load the data for this assignment
DATA_ZIP = WORKDIR + '/gaitData.zip'
DATA_DIRECTORY = WORKDIR + '/gaitData'

if not os.path.isdir(DATA_DIRECTORY):
    import zipfile
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall(os.path.dirname(DATA_ZIP))

## Reading Data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIRECTORY = 'gaitData'
SPEEDS = [25, 35, 45]                 # column suffixes
SPEED_MS = {25: 2.5, 35: 3.5, 45: 4.5}

info = pd.read_csv(os.path.join(DATA_DIRECTORY, 'RBDSinfo.csv'), dtype={'SubjectID': str})
info.head()

---
## Part A — Who is in the study? (1 point)

Use `RBDSinfo.csv`.

**A1.** How many participants are there? How many are male and female?

In [ ]:
# A1
n_participants = len(info)
print('number of participants:', n_participants)
print(info['Gender'].value_counts())

*Your answer for A1:* There are 28 participants: 27 male and 1 female.

**A2.** Find the mean, standard deviation, minimum and maximum of `Age`, `Height`, `Weigth` and `BMI`.

*(Yes — the weight column is spelled `Weigth` in the file. Use the name as it appears.)*

In [ ]:
# A2
cols = ['Age', 'Height', 'Weigth', 'BMI']
info[cols].agg(['mean', 'std', 'min', 'max']).round(2)

*Your answer for A2:* Age 34.75 ± 6.69 years (22–51), height 175.96 ± 6.80 cm (162.7–187.2), weight 69.64 ± 7.74 kg (56.85–82.15), BMI 22.44 ± 1.62 (18.92–25.86). The values are mean ± SD (min–max).

**A3.** Split the participants into injured and uninjured using the `Injury` column, and report the same four statistics for each group. Do you see any differences?

In [ ]:
# A3
print(info['Injury'].value_counts())

# one column per group (No / Yes); .T puts the statistics in rows so the table is easier to read
info.groupby('Injury')[cols].agg(['mean', 'std', 'min', 'max']).T.round(2)

*Your answer for A3:* There are 9 injured and 19 uninjured participants. The two groups are very similar:

- **Weight** is almost identical (69.58 vs 69.66 kg).
- **Age** is close (33.8 vs 35.2 years).
- **Height:** the injured group is about 4.6 cm taller (179.1 vs 174.5 cm).
- **BMI:** the injured group is about 1.2 lower (21.7 vs 22.8), which follows from being taller at the same weight.

With only 9 injured runners these differences are small and could easily be due to chance. Nothing here suggests that age or body size explains who got injured.

**A4.** Which body side is injured most often — left, right or bilateral? Do not count by hand: write code that reads `InjuryLocation` and counts for you. (Watch for messy spelling in that column.)

In [ ]:
# A4
# clean the text first: drop the uninjured (NaN), remove extra spaces, make everything lower case
locations = info['InjuryLocation'].dropna().str.strip().str.lower()

def body_side(location):
    """Return 'bilateral', 'left' or 'right' from a free-text injury location."""
    if 'bil' in location:          # catches 'bilateral' and the misspelled 'bileteral'
        return 'bilateral'
    elif 'left' in location:
        return 'left'
    elif 'right' in location:
        return 'right'
    else:
        return 'unknown'

sides = locations.apply(body_side)
print(pd.DataFrame({'InjuryLocation': info['InjuryLocation'].dropna(), 'side': sides}))
print()
print(sides.value_counts())

*Your answer for A4:* Left and right are tied: 4 left-side injuries and 4 right-side injuries, plus 1 bilateral (both ankles). Neither side is injured more often than the other in this sample. The code catches the messy spellings by stripping spaces, converting to lower case, and searching for `'bil'` so that the misspelled "bileteral" still counts as bilateral.

---
## Part B — One participant, and how speed changes the motion (2 points)

Read one participant's file, for example:

```python
gait = pd.read_csv('gaitData/RBDS001processed.txt', sep='\t')
```

**B1.** For the **right** side, make a figure with **one panel per joint** (hip, knee, ankle). In each panel plot the **sagittal angle (`AngZ`)** at all three speeds, as three lines in different colours on the same axes.

Titles, axis labels and a legend are required. Then answer: as speed goes up, how does each joint's curve change?

In [ ]:
# B1
gait = pd.read_csv(os.path.join(DATA_DIRECTORY, 'RBDS001processed.txt'), sep='\t')

JOINTS = ['hip', 'knee', 'ankle']
SPEED_COLORS = {25: 'tab:blue', 35: 'tab:orange', 45: 'tab:red'}

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for i, joint in enumerate(JOINTS):
    for speed in SPEEDS:
        ax[i].plot(gait['PercGcycle'], gait[f'R{joint}AngZ{speed}'],
                   color=SPEED_COLORS[speed], lw=2, label=f'{SPEED_MS[speed]} m/s')
    ax[i].set_title(f'Right {joint}: sagittal angle')
    ax[i].set_xlabel('Gait cycle (%)')
    ax[i].set_ylabel('Angle (deg)')
    ax[i].legend()
    ax[i].grid()
fig.suptitle('Participant RBDS001: sagittal joint angles at three running speeds')
plt.tight_layout()
plt.show()

*Your answer for B1:* The overall shape of each curve stays the same at every speed, but the movement gets larger as speed increases:

- **Hip:** the range of motion grows a lot, from about 46° at 2.5 m/s to 58° at 3.5 m/s and 72° at 4.5 m/s. The hip flexes more in swing (peak about 46° → 53° → 62°) and also extends further at the end of stance (minimum about 0° → −5° → −10°).
- **Knee:** the biggest change is in the swing phase. The swing flexion peak rises from about 85° to 100° to 116° and arrives slightly earlier in the cycle. The smaller knee flexion during stance (around 15% of the cycle) changes much less.
- **Ankle:** this joint changes the least. The dorsiflexion peak in stance stays at about 22–24°, and plantarflexion at toe-off increases a little (about −27° → −31°). Toe-off also happens earlier in the cycle at higher speed, because stance takes up a smaller part of the stride.

**B2.** Turn that picture into a number. For the **hip** and the **knee** (right, sagittal), find the **peak flexion** — the maximum of `AngZ` — at each of the three speeds. Put the six values in a small table, then plot peak flexion against speed (in m/s). Does peak flexion rise with speed?

In [ ]:
# B2
peak_flexion = pd.DataFrame(
    {joint: [gait[f'R{joint}AngZ{speed}'].max() for speed in SPEEDS] for joint in ['hip', 'knee']},
    index=[SPEED_MS[speed] for speed in SPEEDS])
peak_flexion.index.name = 'speed (m/s)'
display(peak_flexion.round(1))

plt.figure(figsize=(6, 4))
plt.plot(peak_flexion.index, peak_flexion['hip'],  'o-', lw=2, label='Right hip')
plt.plot(peak_flexion.index, peak_flexion['knee'], 's-', lw=2, label='Right knee')
plt.xticks([2.5, 3.5, 4.5])
plt.xlabel('Running speed (m/s)')
plt.ylabel('Peak flexion (deg)')
plt.title('RBDS001: peak sagittal flexion vs speed')
plt.legend()
plt.grid()
plt.show()

*Your answer for B2:* Yes, peak flexion rises with speed at both joints, and almost linearly. The hip goes from 45.9° to 53.1° to 62.0° (about 8° more per 1 m/s). The knee goes from 84.6° to 99.6° to 115.5° (about 15° more per 1 m/s). The knee increases about twice as fast as the hip. Both peaks happen during swing, when the leg is brought forward faster at higher speeds.

**B3.** For the **right** side, make a figure with one panel per joint showing **joint power** (`Pow`) through the gait cycle, with the three speeds overlaid. Which joint produces the most power in early stance? In late stance? Is the power related to running speed?

In [ ]:
# B3
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for i, joint in enumerate(JOINTS):
    for speed in SPEEDS:
        ax[i].plot(gait['PercGcycle'], gait[f'R{joint}Pow{speed}'],
                   color=SPEED_COLORS[speed], lw=2, label=f'{SPEED_MS[speed]} m/s')
    ax[i].axhline(0, color='k', lw=0.8)          # above 0 = power generated, below 0 = power absorbed
    ax[i].set_title(f'Right {joint}: joint power')
    ax[i].set_xlabel('Gait cycle (%)')
    ax[i].set_ylabel('Power (W/kg)')
    ax[i].legend()
    ax[i].grid()
fig.suptitle('Participant RBDS001: sagittal joint power at three running speeds')
plt.tight_layout()
plt.show()

# peak power generated (max) and absorbed (min) by each joint at each speed
pd.DataFrame({f'{joint} {kind}': [getattr(gait[f'R{joint}Pow{speed}'], kind)() for speed in SPEEDS]
              for joint in JOINTS for kind in ['max', 'min']},
             index=pd.Index([SPEED_MS[s] for s in SPEEDS], name='speed (m/s)')).round(1)

*Your answer for B3:* In this participant, stance lasts from 0% to about 32–38% of the gait cycle (ground contact), and it gets shorter as speed increases.

- **Early stance (about 0–15%):** the **knee** has the most power, but it is negative: the knee *absorbs* energy (about −7.4, −9.4 and −12.2 W/kg at the three speeds). The quadriceps work eccentrically to control knee flexion as the body lands. The ankle also absorbs energy at the same time, almost as much at the fastest speed (about −5.3, −8.4 and −11.4 W/kg).
- **Late stance (about 20–35%):** the **ankle** produces the most positive power, about 7.1, 9.5 and 12.2 W/kg. The plantarflexors push the body off the ground. The knee generates some power at mid-stance (about 5–7 W/kg), and the hip's power is smaller than both.
- **Relation to speed:** yes. At all three joints, both the generated and the absorbed power peaks get larger as speed increases. The ankle push-off peak grows the most, and it is the main source of propulsion.

---
## Part C — All 28 participants (2 points)

**C1.** With a `for` loop, read every participant's file. (If you get stuck, `Assignment-No5-Extra.ipynb` shows one way. Try it yourself first.)

In [ ]:
# C1
files = sorted(f for f in os.listdir(DATA_DIRECTORY) if f.endswith('processed.txt'))

all_gait = {}                      # subject ID -> DataFrame with that participant's data
for file in files:
    subject_id = file[4:7]         # 'RBDS001processed.txt' -> '001'
    all_gait[subject_id] = pd.read_csv(os.path.join(DATA_DIRECTORY, file), sep='\t')

print('files read:', len(all_gait))
print('shape of each file:', {df.shape for df in all_gait.values()})

**C2.** Make a figure with **one panel per speed** (2.5, 3.5, 4.5 m/s). In each panel draw the **right knee sagittal angle (`RkneeAngZ`)** for all 28 participants — one line each, use `alpha` so the lines are readable. Is the effect of speed consistent from person to person?

In [ ]:
# C2
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for i, speed in enumerate(SPEEDS):
    for subject_id, df in all_gait.items():
        ax[i].plot(df['PercGcycle'], df[f'RkneeAngZ{speed}'], color=SPEED_COLORS[speed], alpha=0.3)
    ax[i].set_title(f'{SPEED_MS[speed]} m/s (n = {len(all_gait)})')
    ax[i].set_xlabel('Gait cycle (%)')
    ax[i].set_ylabel('Right knee angle (deg)')
    ax[i].grid()
fig.suptitle('Right knee sagittal angle, all participants (one line per participant)')
plt.tight_layout()
plt.show()

# check the effect of speed person by person: peak knee flexion at each speed
knee_peaks = pd.DataFrame({SPEED_MS[s]: [df[f'RkneeAngZ{s}'].max() for df in all_gait.values()] for s in SPEEDS},
                          index=list(all_gait.keys()))
increasing = (knee_peaks[3.5] > knee_peaks[2.5]) & (knee_peaks[4.5] > knee_peaks[3.5])
print(f'participants whose peak knee flexion rises at every speed step: {increasing.sum()} of {len(knee_peaks)}')

*Your answer for C2:* Yes, the effect of speed is consistent from person to person. Every participant has the same curve shape at every speed: a small knee flexion peak in stance, then a large flexion peak in swing. The participants are offset from each other by individual differences: the SD across participants averages about 8° over the cycle. When speed increases, the whole bundle of swing peaks moves upward. Peak knee flexion rises at each speed step for all 28 of 28 participants, so the group change is not driven by just a few runners.

**C3.** Now summarise it. On a **single axis**, plot the group **mean** right knee sagittal angle at each speed, with a shaded **±1 SD** band around each mean. Describe how the mean curve moves as speed increases.

In [ ]:
# C3
plt.figure(figsize=(10, 5))
for speed in SPEEDS:
    # one column per participant, one row per % of the gait cycle
    knee = pd.DataFrame({sid: df[f'RkneeAngZ{speed}'] for sid, df in all_gait.items()})
    mean = knee.mean(axis=1)
    sd   = knee.std(axis=1)
    pgc  = all_gait['001']['PercGcycle']
    plt.plot(pgc, mean, color=SPEED_COLORS[speed], lw=2, label=f'{SPEED_MS[speed]} m/s')
    plt.fill_between(pgc, mean - sd, mean + sd, color=SPEED_COLORS[speed], alpha=0.2)
    print(f'{SPEED_MS[speed]} m/s: peak of the mean curve = {mean.max():.1f} deg at {pgc[mean.idxmax()]:.0f}% of the cycle')

plt.xlabel('Gait cycle (%)')
plt.ylabel('Right knee angle (deg)')
plt.title(f'Right knee sagittal angle: group mean ± 1 SD (n = {len(all_gait)})')
plt.legend()
plt.grid()
plt.show()

*Your answer for C3:* As speed increases, the mean curve keeps the same shape but rises during swing. The peak of the mean knee flexion goes from about 93° at 2.5 m/s to 108° at 3.5 m/s and 119° at 4.5 m/s, and it occurs slightly earlier in the cycle (69% → 67%). The stance part of the curve (0 to about 35%) barely changes: the stance flexion peak only rises from about 43° to 44°. The curve also leaves stance and starts its swing rise earlier at higher speeds, because ground contact takes up a smaller share of the stride. The ±1 SD bands are similar in width at all three speeds (about 8°). In swing they separate clearly, so the speed effect is larger than the variation between people.

**C4.** For every participant, find the **peak right ankle power** (`RanklePow`) at each speed. Draw a **boxplot** of peak ankle power with one box per speed. Does peak ankle power increase with speed?

In [ ]:
# C4
peak_ankle_power = pd.DataFrame(
    {SPEED_MS[s]: [df[f'RanklePow{s}'].max() for df in all_gait.values()] for s in SPEEDS},
    index=list(all_gait.keys()))
display(peak_ankle_power.describe().round(2))

plt.figure(figsize=(7, 5))
plt.boxplot(peak_ankle_power.values)
plt.xticks([1, 2, 3], [f'{s} m/s' for s in peak_ankle_power.columns])
plt.xlabel('Running speed')
plt.ylabel('Peak right ankle power (W/kg)')
plt.title(f'Peak right ankle power by speed (n = {len(peak_ankle_power)})')
plt.grid(axis='y')
plt.show()

increasing = (peak_ankle_power[3.5] > peak_ankle_power[2.5]) & (peak_ankle_power[4.5] > peak_ankle_power[3.5])
print(f'participants whose peak ankle power rises at every speed step: {increasing.sum()} of {len(peak_ankle_power)}')

*Your answer for C4:* Yes, peak ankle power increases clearly with speed. The median goes from about 6.0 W/kg at 2.5 m/s to 8.2 W/kg at 3.5 m/s and 11.1 W/kg at 4.5 m/s (means 5.97, 8.46 and 11.09 W/kg). The boxes barely overlap, and the whole box moves up at each speed. The spread also grows a little with speed (SD 1.2 → 1.4 → 1.6 W/kg). The increase holds for every individual: all 28 of 28 participants produce more peak ankle power at each faster speed. This fits the ankle plantarflexors being the main source of push-off power in running.

---
## What to hand in

One notebook, run top to bottom with a clean kernel so every figure and number
shows. Keep your written answers in the markdown cells.
